<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-14_March-17-2026/Assignment-3-4_Comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 3 - Deadline 2/27/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Add your name here

**AI usage statement:**
Here you should give a statement about any usage of AI tools to assist you with the coding.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Split the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

#### B)
For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

#### C)
Perform classifcation using random forest classification model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: 2
- Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:
- Accuracy
- Precision
- Receiver Operating Characteristic Area Under the Curve (ROC AUC)  

In your analysis and metrics, the high permeability class should be consider as the postive class.

Based on your analysis, which feature set gives the best results for the classifcation?

#### D) - Optional for 2 points
Repeat your analysis from C) using a $k$ nearest neighbors classifer (i.e., $k$ nearest neighbors vote). Use the default value of 5 neighbors.

Do you obtain the same results as in C)?



In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

In [ ]:
%%bash
dataset_url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv"
wget ${dataset_url}
ls

In [ ]:
##capture
!pip install rdkit mols2grid lightgbm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("model_data.csv")
data_smi = pd.read_csv("2d_features.csv")

In [ ]:
data_smi

In [ ]:
data['P_app'] = data_smi['P_app']
data['Smiles'] = data_smi['Smiles']
data['Compound'] = data_smi['Compound']

In [ ]:
Permeable_cutoff = 7.0
Low_label = 0
High_label = +1
Permeable_key_str = f'Permeability High({High_label:})/Low({Low_label:})'
data[Permeable_key_str] = [High_label if p > Permeable_cutoff else Low_label for p in data['P_app']]

Number_Permeable_High = np.sum(data[Permeable_key_str] == +1)
Number_Permeable_Low = np.sum(data[Permeable_key_str] == 0)

print("Key:",Permeable_key_str)

print("Number with high permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_High))
print("Number with low permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_Low))

print("")

data[['P_app', Permeable_key_str] ]

In [ ]:
keys_3d = [
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]

keys_2d = [
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]

keys_combined = keys_2d+keys_3d

features_2d = data[keys_2d]
features_3d = data[keys_3d]
features_combined = data[keys_combined]

In [ ]:
import mols2grid

mols2grid.display(
    data,
    smiles_col = "Smiles",
    subset     = ["img", "Compound", "P_app", "Permeability High(1)/Low(0)"],
    tooltip    = ["Compound", "P_app", "Permeability High(1)/Low(0)"]
)

In [ ]:
import mols2grid

def P_app(x):
  return f'{x:.2f} nm/s'

def permeability_label(x):
  if int(x)==1:
    return "High"
  else:
    return "Low"

mols2grid.display(
    data,
    smiles_col = "Smiles",
    subset     = ["img", "Compound", "P_app", "Permeability High(1)/Low(0)"],
    tooltip    = keys_combined,
    transform={'P_app': P_app,
                "Permeability High(1)/Low(0)": permeability_label
               }
)

In [ ]:
target_classifcation = data['Permeability High(1)/Low(0)']
features_2d = data[keys_2d]
features_3d = data[keys_3d]
features_combined = data[keys_combined]

features_set = {'2D': features_2d,
                '3D': features_3d,
                'Combined': features_combined}





In [ ]:
for f in features_set:


In [ ]:
def do_knn_cv(features,target, n_neighbors=5, do_scaling=True, do_cv_fold=False, do_cv_random=True):
  from sklearn.neighbors import KNeighborsClassifier
  from sklearn import metrics
  from sklearn.model_selection import cross_validate,ShuffleSplit,StratifiedKFold
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import StandardScaler

  if do_scaling:
    model = Pipeline(
        steps=[("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=n_neighbors))]
    )
  else:
    model = KNeighborsClassifier(n_neighbors=n_neighbors)


  scoring = {'accuracy':'accuracy',
              'recall': metrics.make_scorer(metrics.recall_score, zero_division=np.nan),
              'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
              'roc_auc': 'roc_auc'
  }

  if do_cv_fold:
    # employ 5-fold CV
    scores_fold = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=4, shuffle=True),
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  if do_cv_random:
    NumSplits=100
    cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.5)
    scores_random = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=cv_random,
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  print("Accuracy - Test")
  if do_cv_fold:   print("- 5-Fold CV                   : {:.3f} +- {:.3f}".format(scores_fold['test_accuracy'].mean(),scores_fold['test_accuracy'].std()))
  if do_cv_random: print("- Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

  print("ROC AUC - Test")
  if do_cv_fold:   print("- 5-Fold CV                   : {:.3f} +- {:.3f}".format(scores_fold['test_roc_auc'].mean(),scores_fold['test_roc_auc'].std()))
  if do_cv_random: print("- Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

  print("Precision")
  if do_cv_fold:   print("- 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_precision']),np.nanstd(scores_fold['test_precision'])))
  if do_cv_random: print("- Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

In [ ]:
for f in features_set:
  print("Features: {:s}".format(f))

  do_knn_cv(features_set[f],target_classifcation, do_scaling=True)
  print("----------------------------------")
  print("")

In [ ]:
for f in features_set:
  print("Features: {:s}".format(f))

  do_knn_cv(features_set[f],target_classifcation, do_scaling=False)
  print("----------------------------------")
  print("")

In [ ]:
def do_rfregressor_cv(features, target, do_cv_fold=False, do_cv_random=True, show_trainingset=False):
  from sklearn.ensemble import RandomForestRegressor
  from sklearn.model_selection import train_test_split
  from sklearn import metrics
  from sklearn.model_selection import cross_validate,ShuffleSplit


  model = RandomForestRegressor()

  scoring =['neg_mean_absolute_error',
            'neg_mean_squared_error',
            'neg_root_mean_squared_error',
            'neg_max_error',
            'r2']

  if do_cv_fold:
    # employ 5-fold CV
    scores_fold = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=4, shuffle=True),
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  if do_cv_random:
    NumSplits=100
    cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.5)
    scores_random = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=cv_random,
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  for s in scoring:
    print("Score: {:s}".format(s))
    if show_trainingset:
      print("- Training set: {:s}".format(s))
      if do_cv_fold:   print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
      if do_cv_random: print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
      print("- Test set: {:s}".format(s))
    if do_cv_fold:   print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
    if do_cv_random: print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
    print("")

In [ ]:
target_regression = data['P_appLog']

for f in features_set:
  print("Features: {:s}".format(f))

  do_rfregressor_cv(features_set[f],target_regression)
  print("----------------------------------")
  print("")

In [ ]:
target_regression = data['P_app']

for f in features_set:
  print("Features: {:s}".format(f))

  do_rfregressor_cv(features_set[f],target_regression)
  print("----------------------------------")
  print("")

In [ ]:
def do_lightgbmregressor_cv(features, target, do_cv_fold=False, do_cv_random=True, show_trainingset=False):
  from lightgbm import LGBMRegressor, plot_importance
  from sklearn.model_selection import train_test_split
  from sklearn import metrics
  from sklearn.model_selection import cross_validate,ShuffleSplit


  model = LGBMRegressor(verbose=-1)

  scoring =['neg_mean_absolute_error',
            'neg_mean_squared_error',
            'neg_root_mean_squared_error',
            'neg_max_error',
            'r2']

  if do_cv_fold:
    # employ 5-fold CV
    scores_fold = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=4, shuffle=True),
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  if do_cv_random:
    NumSplits=100
    cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.5)
    scores_random = cross_validate(
        model,
        features, target,
        scoring=scoring,
        cv=cv_random,
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

  for s in scoring:
    print("Score: {:s}".format(s))
    if show_trainingset:
      print("- Training set: {:s}".format(s))
      if do_cv_fold:   print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
      if do_cv_random: print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
      print("- Test set: {:s}".format(s))
    if do_cv_fold:   print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
    if do_cv_random: print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
    print("")

In [ ]:
target_regression = data['P_appLog']

for f in features_set:
  print("Features: {:s}".format(f))

  do_lightgbmregressor_cv(features_set[f],target_regression)
  print("----------------------------------")
  print("")

In [ ]:
target_regression = data['P_app']

for f in features_set:
  print("Features: {:s}".format(f))

  do_lightgbmregressor_cv(features_set[f],target_regression)
  print("----------------------------------")
  print("")